Y


In [1]:
# === CONFIG: edit these two paths ===
SI_FOLDER = r"./SI downloaded"  # folder containing files like 10.1002_adma.202210613_SI.pdf or .docx
EXCEL_PATH = r"./SELECTED 7000 SI - Copy.xlsx"  # the workbook with DOI and SI Downloaded columns
# NEW: folder for main articles, files like 10.1039_d4ra00914b.pdf
DOWNLOADED_FOLDER = r"./downloaded"
# ====================================

from pathlib import Path
import pandas as pd
import numpy as np
import re

si_dir = Path(SI_FOLDER)
excel_path = Path(EXCEL_PATH)
main_dir = Path(DOWNLOADED_FOLDER)  # NEW: main-article folder

if not si_dir.is_dir():
    raise FileNotFoundError(f"SI folder not found: {si_dir.resolve()}")
if not excel_path.is_file():
    raise FileNotFoundError(f"Excel file not found: {excel_path.resolve()}")
# NEW: allow the main folder to be optional but warn if missing
if not main_dir.is_dir():
    print(f'Warning: main "downloaded" folder not found at: {main_dir.resolve()}')
    print("Main article matching will be skipped.\n")

# Collect available filenames in the SI folder (case-insensitive lookup)
allowed_exts = {".pdf", ".docx", ".doc"}
file_lookup = {}
for p in si_dir.iterdir():
    if p.is_file() and p.suffix.lower() in allowed_exts:
        file_lookup[p.name.lower()] = p.name  # map lower to original

# NEW: Collect available filenames in the main-article folder (pdf only, case-insensitive)
main_lookup = {}
if main_dir.is_dir():
    for p in main_dir.iterdir():
        if p.is_file() and p.suffix.lower() == ".pdf":
            main_lookup[p.name.lower()] = p.name

# Load Excel
df = pd.read_excel(excel_path)

# Normalize column names and ensure required columns exist
df.columns = [str(c).strip() for c in df.columns]
if "DOI" not in df.columns:
    raise KeyError('Column "DOI" not found in the sheet.')
if "SI Downloaded" not in df.columns:
    df["SI Downloaded"] = ""
if "Downloaded" not in df.columns:
    df["Downloaded"] = ""

# Force writable string-friendly dtypes to avoid ValueError on ""
df["SI Downloaded"] = df["SI Downloaded"].astype(object)
df["Downloaded"] = df["Downloaded"].astype(object)

# Helper to treat empty cells consistently
def is_empty(x):
    if pd.isna(x):
        return True
    s = str(x).strip()
    return s == "" or s.lower() in {"nan", "none"}

# Normalize a DOI string to the base used in filenames
def doi_to_base(doi_raw: str) -> str:
    # strip common prefixes like https://doi.org/ or doi:
    doi = str(doi_raw).strip()
    doi = re.sub(r'^(?:https?://)?(?:dx\.)?doi\.org/', "", doi, flags=re.IGNORECASE)
    doi = re.sub(r'^doi:\s*', "", doi, flags=re.IGNORECASE)
    doi = doi.strip()
    # file pattern uses underscores instead of the single slash in a DOI
    # example: 10.1002/adma.202210613 -> 10.1002_adma.202210613
    return doi.replace("/", "_")

# Prepare output paths
out_path = excel_path.with_name(f"{excel_path.stem} - updated{excel_path.suffix}")
simple_out_path = excel_path.with_name(f"{excel_path.stem} - simple.xlsx")

# Optional: also record which file matched
found_si_col = "Found SI Filename"
if found_si_col not in df.columns:
    df[found_si_col] = ""
df[found_si_col] = df[found_si_col].astype(object)

# NEW: record which main file matched
found_main_col = "Matched Main Filename"
if found_main_col not in df.columns:
    df[found_main_col] = ""
df[found_main_col] = df[found_main_col].astype(object)

# NEW: full paths for simpler export
si_path_col = "SI File"
main_path_col = "Main File"
if si_path_col not in df.columns:
    df[si_path_col] = ""
if main_path_col not in df.columns:
    df[main_path_col] = ""
df[si_path_col] = df[si_path_col].astype(object)
df[main_path_col] = df[main_path_col].astype(object)

# Do the updates for EVERY row
updated_si = 0
updated_main = 0
rows_total = len(df)

for idx in df.index:
    doi_val = df.at[idx, "DOI"]
    if is_empty(doi_val):
        # Clear outputs if DOI missing
        df.at[idx, "SI Downloaded"] = ""
        df.at[idx, "Downloaded"] = ""
        df.at[idx, found_si_col] = ""
        df.at[idx, found_main_col] = ""
        df.at[idx, si_path_col] = ""
        df.at[idx, main_path_col] = ""
        continue

    base = doi_to_base(doi_val)

    # SI candidates: check in order
    si_candidates = [
        f"{base}_SI.pdf",
        f"{base}_SI.docx",
        f"{base}_SI.doc",
    ]
    si_matched_name = None
    for cand in si_candidates:
        if cand.lower() in file_lookup:
            si_matched_name = file_lookup[cand.lower()]
            break

    if si_matched_name is not None:
        df.at[idx, "SI Downloaded"] = 1
        df.at[idx, found_si_col] = si_matched_name
        df.at[idx, si_path_col] = str(si_dir / si_matched_name)
        updated_si += 1
    else:
        df.at[idx, "SI Downloaded"] = ""
        df.at[idx, found_si_col] = ""
        df.at[idx, si_path_col] = ""

    # Main candidate: only one pattern for main
    main_matched_name = None
    if main_dir.is_dir():
        main_cand = f"{base}.pdf"
        if main_cand.lower() in main_lookup:
            main_matched_name = main_lookup[main_cand.lower()]

    if main_matched_name is not None:
        df.at[idx, "Downloaded"] = 1
        df.at[idx, found_main_col] = main_matched_name
        df.at[idx, main_path_col] = str(main_dir / main_matched_name)
        updated_main += 1
    else:
        df.at[idx, "Downloaded"] = ""
        df.at[idx, found_main_col] = ""
        df.at[idx, main_path_col] = ""

# Save the full workbook
df.to_excel(out_path, index=False)

# Report using the final DataFrame so the numbers reconcile
def is_one(x):
    try:
        return str(int(float(x))).strip() == "1"
    except Exception:
        return str(x).strip() == "1"

si_true = df["SI Downloaded"].apply(is_one)
main_true = df["Downloaded"].apply(is_one)

both_count = int((si_true & main_true).sum())
only_main_count = int((~si_true & main_true).sum())
only_si_count = int((si_true & ~main_true).sum())
neither_count = int((~si_true & ~main_true).sum())

print("Done.")
print(f"Workbook saved to: {out_path.name}")
print(f"Total rows: {rows_total}")
print(f"SI updated to 1: {updated_si}")
print(f"Main updated to 1: {updated_main}")
print("\nSummary:")
print(f"Both main and SI present: {both_count}")
print(f"Only main article present: {only_main_count}")
print(f"Only SI present: {only_si_count}")
print(f"Neither present: {neither_count}")

# NEW: list any files in folders that do not match any DOI in the sheet
# Build set of normalized DOI bases from the sheet
doi_bases = set()
for v in df["DOI"]:
    if not is_empty(v):
        doi_bases.add(doi_to_base(v))

# Collect SI bases from filenames: <base>_SI.<ext>
si_unmatched = []
for fname_lower, fname_orig in file_lookup.items():
    stem = Path(fname_lower).stem  # removes extension
    if stem.lower().endswith("_si"):
        base = stem[:-3]  # drop "_SI"
        if base not in doi_bases:
            si_unmatched.append(fname_orig)
    else:
        if stem not in doi_bases:
            si_unmatched.append(fname_orig)

main_unmatched = []
for fname_lower, fname_orig in main_lookup.items():
    stem = Path(fname_lower).stem  # like 10.1039_d4ra00914b
    if stem not in doi_bases:
        main_unmatched.append(fname_orig)

print("\nUnmatched files:")
print(f"SI files not matched to any DOI in sheet: {len(si_unmatched)}")
for x in si_unmatched:
    print(f"  SI unmatched: {x}")

print(f"Main files not matched to any DOI in sheet: {len(main_unmatched)}")
for x in main_unmatched:
    print(f"  Main unmatched: {x}")

# NEW: write a simple 3-column file for downstream use
simple_df = df.loc[:, ["DOI", main_path_col, si_path_col]].rename(
    columns={"DOI": "DOI", main_path_col: "Main File", si_path_col: "SI File"}
)
simple_df.to_excel(simple_out_path, index=False)
print(f"\nSimple file saved to: {simple_out_path.name} (columns: DOI, Main File, SI File)")


Done.
Workbook saved to: SELECTED 7000 SI - Copy - updated.xlsx
Total rows: 7437
SI updated to 1: 5423
Main updated to 1: 6165

Summary:
Both main and SI present: 4678
Only main article present: 1487
Only SI present: 745
Neither present: 527

Unmatched files:
SI files not matched to any DOI in sheet: 16
  SI unmatched: 10.1002_adfm.202410751_SI.docx
  SI unmatched: 10.1016_S1872-2067(23)64562-0_SI.pdf
  SI unmatched: 10.1016_S1872-2067(24)60020-3_SI.pdf
  SI unmatched: 10.1021_acs.chemmater.9b02322_SI.pdf
  SI unmatched: 10.1021_jacs.5c08726_SI.pdf
  SI unmatched: 10.1038_NCHEM.1003_SI.pdf
  SI unmatched: 10.1038_NCHEM.1982_SI.pdf
  SI unmatched: 10.1038_NCHEM.2430_SI.pdf
  SI unmatched: 10.1038_NCHEM.254_SI.pdf
  SI unmatched: 10.1038_NCHEM.738_SI.pdf
  SI unmatched: 10.1038_NCHEM.834_SI.pdf
  SI unmatched: 10.1038_NMAT4113_SI.pdf
  SI unmatched: 10.1038_NMAT5050_SI.pdf
  SI unmatched: 10.1038_s41467-020-18968-7_SI.pdf
  SI unmatched: 10.1038_s41467-025-64092-9_SI.pdf
  SI unmatched: 


Simple file saved to: SELECTED 7000 SI - Copy - simple.xlsx (columns: DOI, Main File, SI File)


In [ ]:
# === CELL 2: compute OpenAI-compatible tokens and word counts, update sheet, and plot ===
import time
import re
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Require tiktoken for OpenAI-compatible token counting
try:
    import tiktoken
except Exception as e:
    raise ImportError(
        "tiktoken is required for OpenAI-compatible token counts. "
        "Install with: pip install --upgrade tiktoken"
    ) from e

# Try to import a robust PDF extractor
pdf_backend = None
try:
    import pdfminer.high_level as pdfminer_high
    pdf_backend = "pdfminer"
except Exception:
    try:
        import PyPDF2
        pdf_backend = "pypdf2"
    except Exception:
        pdf_backend = None

# Use same naming as in first cell
UPDATED_EXCEL = excel_path.with_name(f"{excel_path.stem} - updated{excel_path.suffix}")

# Column names expected from the first cell
MAIN_FILE_COL = "Main File"
SI_FILE_COL = "SI File"

# Output metric columns
MAIN_WORDS = "Main Words"
SI_WORDS = "SI Words"
COMBINED_WORDS = "Combined Words"

MAIN_TOKENS = "Main Tokens"
SI_TOKENS = "SI Tokens"
COMBINED_TOKENS = "Combined Tokens"

# Configure tokenizer: use GPT-4o encoding or fall back to cl100k_base
try:
    encoding = tiktoken.encoding_for_model("gpt-4o")
except Exception:
    encoding = tiktoken.get_encoding("cl100k_base")

_word_re = re.compile(r"\b\w+\b", flags=re.UNICODE)

def read_pdf_text(path: Path) -> str:
    """Extract text from a PDF file. Returns empty string on failure."""
    try:
        if pdf_backend == "pdfminer":
            return pdfminer_high.extract_text(str(path)) or ""
        elif pdf_backend == "pypdf2":
            import PyPDF2
            text_parts = []
            with open(path, "rb") as f:
                reader = PyPDF2.PdfReader(f)
                for page in reader.pages:
                    try:
                        text_parts.append(page.extract_text() or "")
                    except Exception:
                        text_parts.append("")
            return "\n".join(text_parts)
        else:
            return ""
    except Exception:
        return ""

def count_words(text: str) -> int:
    if not text:
        return 0
    return len(_word_re.findall(text))

def count_tokens_openai(text: str) -> int:
    """Count tokens using OpenAI-compatible tokenizer."""
    if not text:
        return 0
    return len(encoding.encode(text))

def safe_path(x) -> Optional[Path]:
    if pd.isna(x):
        return None
    s = str(x).strip()
    if not s:
        return None
    p = Path(s)
    if p.is_file():
        return p
    return None

def needs_value(v) -> bool:
    if v is None:
        return True
    if pd.isna(v):
        return True
    s = str(v).strip()
    return s == "" or s.lower() in {"nan", "none"}

def finite_series(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()

def avg_and_total(series: pd.Series):
    s = finite_series(series)
    return (float(s.mean()) if len(s) else 0.0, int(s.sum()) if len(s) else 0)

# Load workbook
df2 = pd.read_excel(UPDATED_EXCEL)

# Ensure required columns exist
for col in [MAIN_FILE_COL, SI_FILE_COL]:
    if col not in df2.columns:
        df2[col] = ""

for col in [MAIN_WORDS, SI_WORDS, COMBINED_WORDS, MAIN_TOKENS, SI_TOKENS, COMBINED_TOKENS]:
    if col not in df2.columns:
        df2[col] = np.nan

# Count PDFs needing processing
total_pdfs_to_process = 0
for _, row in df2.iterrows():
    main_p = safe_path(row.get(MAIN_FILE_COL, ""))
    si_p = safe_path(row.get(SI_FILE_COL, ""))
    if main_p and (needs_value(row.get(MAIN_WORDS)) or needs_value(row.get(MAIN_TOKENS))):
        total_pdfs_to_process += 1
    if si_p and (needs_value(row.get(SI_WORDS)) or needs_value(row.get(SI_TOKENS))):
        total_pdfs_to_process += 1

print(f"PDFs to process based on missing counts: {total_pdfs_to_process}")

# Update missing counts only
processed = 0
start_time = time.time()

for idx in df2.index:
    row = df2.loc[idx]

    # Main
    main_path = safe_path(row.get(MAIN_FILE_COL, ""))
    if main_path and (needs_value(row.get(MAIN_WORDS)) or needs_value(row.get(MAIN_TOKENS))):
        text = read_pdf_text(main_path)
        if needs_value(row.get(MAIN_WORDS)):
            df2.at[idx, MAIN_WORDS] = int(count_words(text))
        if needs_value(row.get(MAIN_TOKENS)):
            df2.at[idx, MAIN_TOKENS] = int(count_tokens_openai(text))
        processed += 1

    # SI
    si_path = safe_path(row.get(SI_FILE_COL, ""))
    if si_path and (needs_value(row.get(SI_WORDS)) or needs_value(row.get(SI_TOKENS))):
        text = read_pdf_text(si_path)
        if needs_value(row.get(SI_WORDS)):
            df2.at[idx, SI_WORDS] = int(count_words(text))
        if needs_value(row.get(SI_TOKENS)):
            df2.at[idx, SI_TOKENS] = int(count_tokens_openai(text))
        processed += 1

    # Progress print every 100 PDFs
    if processed and processed % 100 == 0:
        elapsed = time.time() - start_time
        print(f"Processed {processed}/{total_pdfs_to_process} PDFs in {elapsed:.1f}s")

# Combined columns: use SI if present, else Main
def coalesce(pref, alt):
    if not pd.isna(pref):
        return pref
    return alt

df2[COMBINED_WORDS] = df2[[SI_WORDS, MAIN_WORDS]].apply(
    lambda r: coalesce(r[SI_WORDS], r[MAIN_WORDS]), axis=1
)
df2[COMBINED_TOKENS] = df2[[SI_TOKENS, MAIN_TOKENS]].apply(
    lambda r: coalesce(r[SI_TOKENS], r[MAIN_TOKENS]), axis=1
)

# Save updates
df2.to_excel(UPDATED_EXCEL, index=False)
print(f"Counts updated and saved to: {UPDATED_EXCEL.name}")

# Plot histograms for words
main_words = finite_series(df2[MAIN_WORDS])
si_words = finite_series(df2[SI_WORDS])
combined_words = finite_series(df2[COMBINED_WORDS])

plt.figure()
plt.hist(main_words, bins=50)
plt.title("Main words distribution")
plt.xlabel("Words")
plt.ylabel("Count")
plt.show()

plt.figure()
plt.hist(si_words, bins=50)
plt.title("SI words distribution")
plt.xlabel("Words")
plt.ylabel("Count")
plt.show()

plt.figure()
plt.hist(combined_words, bins=50)
plt.title("Combined words distribution")
plt.xlabel("Words")
plt.ylabel("Count")
plt.show()

# Words averages and totals
mw_avg, mw_sum = avg_and_total(df2[MAIN_WORDS])
sw_avg, sw_sum = avg_and_total(df2[SI_WORDS])
cw_avg, cw_sum = avg_and_total(df2[COMBINED_WORDS])

print("\nWords summary:")
print(f"Average Main: {mw_avg:.2f}, Total Main: {mw_sum}")
print(f"Average SI: {sw_avg:.2f}, Total SI: {sw_sum}")
print(f"Average Combined: {cw_avg:.2f}, Total Combined: {cw_sum}")

# Tokens histograms
main_tokens = finite_series(df2[MAIN_TOKENS])
si_tokens = finite_series(df2[SI_TOKENS])
combined_tokens = finite_series(df2[COMBINED_TOKENS])

plt.figure()
plt.hist(main_tokens, bins=50)
plt.title("Main tokens distribution")
plt.xlabel("Tokens")
plt.ylabel("Count")
plt.show()

plt.figure()
plt.hist(si_tokens, bins=50)
plt.title("SI tokens distribution")
plt.xlabel("Tokens")
plt.ylabel("Count")
plt.show()

plt.figure()
plt.hist(combined_tokens, bins=50)
plt.title("Combined tokens distribution")
plt.xlabel("Tokens")
plt.ylabel("Count")
plt.show()

# Token averages and totals
mt_avg, mt_sum = avg_and_total(df2[MAIN_TOKENS])
st_avg, st_sum = avg_and_total(df2[SI_TOKENS])
ct_avg, ct_sum = avg_and_total(df2[COMBINED_TOKENS])

print("\nTokens summary:")
print(f"Average Main: {mt_avg:.2f}, Total Main: {mt_sum}")
print(f"Average SI: {st_avg:.2f}, Total SI: {st_sum}")
print(f"Average Combined: {ct_avg:.2f}, Total Combined: {ct_sum}")

elapsed_total = time.time() - start_time
print(f"\nAll done. Elapsed time: {elapsed_total:.1f}s")
